# Week #6(a): Random Numbers and Random Grids

[Open this notebook in Colab](https://colab.research.google.com/github/neurosynthetic/grant-vogt/blob/main/notebooks/week06a.ipynb)

> **Important:** Before you do any work, save your own copy to Google Drive.
>
> In Colab: **File → Save a copy in Drive**

In earlier weeks, you transformed grids that were already given to you. This week, you will create new grids using **random numbers**.

We will return to the vertical bars from Week 4. First, only their shared color will vary. Then you will add random heights, a random number of bars, and a random canvas height. Each grid exercise keeps the previous possibilities and adds more.

You will learn `import random`, `random.randint(...)`, and how to check a program whose output can change every time you run it. No previous experience with random numbers is needed.

## Goals

By the end of this lesson, you will be able to:

- generate a random integer within a chosen range
- make a fresh random choice each time a function runs
- decide which choices belong outside a loop and which belong inside it
- build increasingly varied bar grids using a provided drawing function
- check both the structure and the variety of generated grids
- explain why random results can repeat

## Setup

Run the next two collapsed cells once. They provide drawing tools, a bar-drawing function, and the **Check Exercise** buttons.

You will write the random-number code. The drawing and checking code is provided, just as the transformation library was provided in Week 5(b).

In [ ]:
#@title Setup utilities and bar drawing (provided — run once)

import copy
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.colors import ListedColormap

GRID_COLORS = [
    "#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00",
    "#AAAAAA", "#F012BE", "#FF851B", "#7FDBFF", "#870C25",
]
GRID_CMAP = ListedColormap(GRID_COLORS)


def draw_grid(ax, grid, title=""):
    rows = len(grid)
    cols = len(grid[0])
    ax.imshow(grid, cmap=GRID_CMAP, vmin=0, vmax=9, interpolation="nearest")
    font_size = max(6, min(12, 120 // max(rows, cols)))
    for row_index, row in enumerate(grid):
        for column_index, color in enumerate(row):
            text_color = "white" if color in {0, 1, 2, 6, 9} else "black"
            ax.text(column_index, row_index, str(color), ha="center", va="center",
                    color=text_color, fontsize=font_size, fontweight="bold")
    ax.set_title(title)
    ax.set_xticks([x - 0.5 for x in range(1, cols)], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, rows)], minor=True)
    ax.grid(which="minor", color="white", linewidth=1)
    ax.tick_params(which="both", left=False, bottom=False,
                   labelleft=False, labelbottom=False)


def show_grid(grid, title="Grid"):
    _, ax = plt.subplots(figsize=(4, 3))
    draw_grid(ax, grid, title)
    plt.tight_layout()
    plt.show()


def show_samples(generator, count=4):
    # Avoid plotting the incomplete starter's None/empty return value.
    grids = []
    try:
        for sample_index in range(count):
            grid = generator()
            if not isinstance(grid, list) or not grid or not grid[0]:
                print("Finish this generator, then run this preview again.")
                return
            grids.append(copy.deepcopy(grid))
        _, axes = plt.subplots(1, count, figsize=(3 * count, 3), squeeze=False)
        for sample_index, grid in enumerate(grids):
            draw_grid(axes[0][sample_index], grid, f"Sample {sample_index + 1}")
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print("Could not preview this generator:", exc)
        print("Use its Check Exercise button for more detailed feedback.")


def draw_bars(bar_heights, bar_color, grid_height):
    grid = []
    for row_index in range(grid_height):
        new_row = []
        for column_index in range(2 * len(bar_heights) + 1):
            new_row.append(0)
        grid.append(new_row)
    for bar_index in range(len(bar_heights)):
        column_index = 2 * bar_index + 1
        for row_index in range(grid_height - bar_heights[bar_index], grid_height):
            grid[row_index][column_index] = bar_color
    return grid


def make_check_button(check_fn):
    button = widgets.Button(description="Check Exercise", button_style="info")
    output = widgets.Output()

    def on_click(_):
        output.clear_output()
        with output:
            check_fn()

    button.on_click(on_click)
    display(button, output)


print("Drawing utilities ready.")


In [ ]:
#@title Checker utilities (provided — run once)

from collections import Counter

# Three fixed batches make feedback reproducible. Checks restore the
# random module's previous state so previews can continue naturally.
_CHECK_SEEDS = (60601, 60602, 60603)
_SAMPLES_PER_BATCH = 200


def _check(condition, message):
    if not condition:
        raise AssertionError(message)


def _check_balance(values, options, label):
    counts = Counter(values)
    expected = len(values) / len(options)
    _check(set(counts) == set(options),
           f"{label}: some allowed values never appeared. Counts: {dict(counts)}. "
           "Check both endpoints of randint.")
    # Deliberately broad bands: no exact frequency or equal-count requirement.
    for value in options:
        _check(expected * 0.15 <= counts[value] <= expected * 2.5,
               f"{label}: value {value} appeared {counts[value]} times in "
               f"{len(values)} choices. Check for fixed or strongly favored values.")


def _check_short_cycle(values, label):
    for period in range(1, 21):
        repeating = all(value == values[index % period]
                        for index, value in enumerate(values))
        _check(not repeating,
               f"{label}: the same short sequence repeats. Make fresh random choices "
               "inside your generator; do not cycle through a fixed list.")


def _inspect_bar_grid(grid, stage):
    _check(type(grid) is list and len(grid) > 0,
           "Return a nonempty grid: a list of row lists.")
    _check(all(type(row) is list and len(row) > 0 for row in grid),
           "Each grid row must be a nonempty list.")
    canvas_height = len(grid)
    width = len(grid[0])
    _check(all(len(row) == width for row in grid),
           "Every row must have the same width.")
    _check(all(type(value) is int and 0 <= value <= 9 for row in grid for value in row),
           "Every cell must be an integer color from 0 through 9.")
    _check(len({id(row) for row in grid}) == canvas_height,
           "Build a separate list for every row; the rows must not share one list.")

    if stage == 5:
        _check(4 <= canvas_height <= 9, "The canvas height must be 4 through 9.")
    else:
        _check(canvas_height == 6, "This exercise keeps the canvas height at 6.")

    if stage <= 3:
        _check(width == 7, "Three bars, their gaps, and borders require width 7.")
    else:
        _check(width in (3, 5, 7, 9, 11),
               "Use 1 through 5 bars, with width 2 * bar_count + 1.")
    bar_count = (width - 1) // 2
    colors = {value for row in grid for value in row if value != 0}
    _check(len(colors) == 1,
           "Choose one nonzero color for the entire grid, shared by all bars.")
    color = next(iter(colors))

    heights = []
    for column_index in range(width):
        column = [row[column_index] for row in grid]
        if column_index % 2 == 0:
            _check(all(value == 0 for value in column),
                   "Even columns, including both side borders, must stay black.")
            continue
        height = sum(value != 0 for value in column)
        _check(1 <= height <= canvas_height, "Every odd column must contain a bar.")
        expected_column = [0] * (canvas_height - height) + [color] * height
        _check(column == expected_column,
               "Bars must be solid, one cell wide, and reach the bottom row.")
        heights.append(height)

    if stage == 2:
        _check(heights == [2, 4, 1],
               "Only color varies here. Keep the left-to-right heights [2, 4, 1].")
    return color, tuple(heights), canvas_height, bar_count


def _check_grid_variety(records, stage):
    _check_balance([item[0] for item in records], range(1, 10), "Bar colors")
    _check_short_cycle(records, "Generated grids")
    if stage == 2:
        return

    if stage >= 4:
        _check_balance([item[3] for item in records], range(1, 6), "Bar counts")
    if stage == 5:
        _check_balance([item[2] for item in records], range(4, 10), "Canvas heights")

    # Pool bar heights separately for each possible canvas height.
    # Also inspect the first bar separately so later random bars cannot
    # conceal a first bar that is always fixed.
    canvas_heights = range(4, 10) if stage == 5 else (6,)
    for canvas_height in canvas_heights:
        group = [item for item in records if item[2] == canvas_height]
        _check(len(group) > 0, f"No grids with canvas height {canvas_height} appeared.")
        all_heights = [height for item in group for height in item[1]]
        _check_balance(all_heights, range(1, canvas_height + 1),
                       f"Bar heights on a {canvas_height}-row canvas")
        _check(len({item[1][0] for item in group}) >= 3,
               "The first bar needs its own fresh random height, too.")
        _check(len({item[0] for item in group}) >= 4,
               "Choose color separately from the canvas height.")

    # Every bar position must vary, including the later positions.
    last_position = 2 if stage == 3 else 4
    for position in range(last_position + 1):
        seen = [item[1][position] for item in records if item[3] > position]
        _check(len(set(seen)) >= 3,
               f"Bar {position + 1} does not vary enough. Draw its height inside the loop.")

    pairs = [(item[1][0], item[1][1]) for item in records if item[3] >= 2]
    _check(len(pairs) >= 100, "Too few samples have at least two bars.")
    for relation, count in [
        ("shorter than", sum(a < b for a, b in pairs)),
        ("equal to", sum(a == b for a, b in pairs)),
        ("taller than", sum(a > b for a, b in pairs)),
    ]:
        _check(count >= 10,
               f"The first bar should sometimes be {relation} the second. "
               "Choose each height separately; repeats are allowed and no sorting is needed.")

    _check(len(set(records)) >= 150,
           "Too few distinct grids appeared. Make new choices on every call.")

    # A color should occur with several height patterns, not determine them.
    for color in range(1, 10):
        patterns = {item[1] for item in records if item[0] == color}
        _check(len(patterns) >= 8,
               "The heights seem tied to the color. Use separate random choices.")

    if stage >= 4:
        for bar_count in range(1, 6):
            group = [item for item in records if item[3] == bar_count]
            _check(len({item[0] for item in group}) >= 4,
                   "Choose color separately from the number of bars.")
    if stage == 5:
        combinations = {(item[2], item[3]) for item in records}
        _check(len(combinations) == 30,
               "Canvas height and bar count should be separate choices. "
               "Some combinations never appeared.")


def check_random_color(generator):
    import random
    saved_state = random.getstate()
    try:
        values = []
        for seed in _CHECK_SEEDS:
            random.seed(seed)
            for sample_index in range(_SAMPLES_PER_BATCH):
                value = generator()
                _check(type(value) is int and 1 <= value <= 9,
                       "Return an integer from 1 through 9; 0 is the black background.")
                values.append(value)
        _check_balance(values, range(1, 10), "Colors")
        _check_short_cycle(values, "Colors")
    except AssertionError as exc:
        print("Not yet:", exc)
        return False
    except Exception as exc:
        print("Your function raised an error:", exc)
        return False
    finally:
        random.setstate(saved_state)
    print("Exercise 1 passes: 600 valid choices, with a broad spread of colors.")
    return True


def check_bar_generator(generator, stage):
    import random
    saved_state = random.getstate()
    try:
        records = []
        previous_grid = None
        previous_snapshot = None
        for seed in _CHECK_SEEDS:
            random.seed(seed)
            for sample_index in range(_SAMPLES_PER_BATCH):
                grid = generator()
                if previous_grid is not None:
                    _check(previous_grid == previous_snapshot,
                           "A new call changed a previously returned grid. Build a fresh grid.")
                try:
                    records.append(_inspect_bar_grid(grid, stage))
                except AssertionError as exc:
                    raise AssertionError(f"Generated grid {len(records) + 1}: {exc}") from exc
                previous_grid = grid
                previous_snapshot = copy.deepcopy(grid)
        _check_grid_variety(records, stage)
    except AssertionError as exc:
        print("Not yet:", exc)
        return False
    except Exception as exc:
        print("Your generator raised an error:", exc)
        return False
    finally:
        random.setstate(saved_state)
    print(f"Exercise {stage} passes: 600 valid grids with the expected variety.")
    print(f"Different grids observed: {len(set(records))}.")
    return True


print("Checker utilities ready.")


# Part 1: Your first random numbers

An **import** makes a Python library available. The line `import random` gives us tools for making random choices.

`random.randint(1, 6)` returns an integer from `1` through `6`, like rolling a six-sided die. The dot means that `randint` comes from the `random` library.

**Both endpoints are included.** This differs from `range(1, 6)`, whose numbers stop at `5`.

Run the next cell several times. You may see different numbers, but the same number can appear again.

In [ ]:
import random

die_roll = random.randint(1, 6)
print("You rolled:", die_roll)


## Several rolls

Call `random.randint` again whenever you want a fresh choice. This loop rolls the die ten times.

Random does **not** mean “different every time.” Two consecutive rolls can match, and ten rolls do not have to include every possible number.

In [ ]:
for roll_number in range(10):
    print(random.randint(1, 6))


## Exercise 1: Choose a random color

Write `random_color()` so that **each call** returns a fresh random integer from `1` through `9`, with each color equally likely.

Color `0` is black, so it is reserved for the background.

Make the random choice **inside** your function. Choosing a number once outside the function and returning it repeatedly would keep giving the same color.

In [ ]:
def random_color():
    return None


In [ ]:
#@title Check Exercise 1

def check_exercise_1():
    return check_random_color(random_color)


make_check_button(check_exercise_1)


## How can we check a random answer?

There is no single expected grid. Instead, the checkers ask two questions:

1. **Is every result valid?** Check sizes, colors, bar heights, black gaps, and bottom alignment.
2. **Do results show enough variety?** Check that allowed choices appear across many calls, that no choice dominates, and that new properties actually vary.

Each checker makes **600 calls** in three batches. It allows repeated results and uneven counts; it does not demand perfect balance.

The checks use a repeatable random sequence internally, so feedback is consistent for the same implementation. They restore the earlier random state afterward. You do not need to set anything up for this.

These are practical checks for common mistakes. A finite sample cannot prove that a generator is random or that all choices are independent.

# Part 2: Random choices become grids

We will use this provided function, based on the bar drawing from Week 4:

```python
draw_bars(bar_heights, bar_color, grid_height)
```

It draws a new grid. For example, `draw_bars([2, 4, 1], 2, 6)` creates three red bars, with heights 2, 4, and 1, on a six-row canvas.

Every generated grid follows these rules:

- All bars in one grid share one color from `1` through `9`.
- Bars are solid, one cell wide, and reach the bottom row.
- One black column separates neighboring bars, with a black column at each side.
- The width is `2 * number_of_bars + 1`.
- **Equal heights are allowed this week.** We are generating grids, and no sorting is required.
- A bar may reach the top row; a black top border is not required.

Keep using ordinary lists of heights. There is no need to keep a separate description for every bar.

In [ ]:
example_grid = draw_bars([2, 4, 1], 2, 6)
show_grid(example_grid, "Fixed example: three red bars")


## Our growing family of grids

Each stage keeps the earlier random choices and adds one more. Every grid from an earlier grid exercise is still possible in the next exercise.

| Exercise | Shared color | Bar heights | Number of bars | Canvas height |
| --- | --- | --- | --- | --- |
| 2 | Random: 1–9 | Fixed: `[2, 4, 1]` | Fixed: 3 | Fixed: 6 |
| 3 | Random: 1–9 | Each random: 1–6 | Fixed: 3 | Fixed: 6 |
| 4 | Random: 1–9 | Each random: 1–6 | Random: 1–5 | Fixed: 6 |
| 5 | Random: 1–9 | Each random: 1–canvas height | Random: 1–5 | Random: 4–9 |

For now, a grid's bars share one color. Giving each bar its own color would be another possible extension later.

## Exercise 2: Same bars, random color

Write `colorful_bars()` so it returns a grid with:

- heights `[2, 4, 1]`, in that order
- one random color, using your `random_color()` helper
- a canvas height of `6`

Call the provided `draw_bars` function to produce the result.

**Predict:** How many different grids can this generator make?

In [ ]:
def colorful_bars():
    return None


In [ ]:
#@title Check Exercise 2

def check_exercise_2():
    return check_bar_generator(colorful_bars, 2)


make_check_button(check_exercise_2)


In [ ]:
# Rerun this cell to see another set of samples.
# Pass the function's name without parentheses so the preview can call it four times.
show_samples(colorful_bars)


# Part 3: A separate height for each bar

Now keep the random color and also vary the heights.

Choose the shared color **once per grid**. Choose a height **once per bar**, inside the loop. That placement matters:

```python
numbers = []
for number_index in range(3):
    new_number = random.randint(1, 6)
    numbers.append(new_number)
```

This produces three separate random choices. They may happen to match, but one choice does not force the others to match.

## Exercise 3: Random heights

Write `random_height_bars()` so it returns three bars on a six-row canvas.

- Keep the shared random color from Exercise 2.
- Build a list of three heights.
- Choose each height separately from `1` through `6`.
- Call `draw_bars` with that list, the color, and canvas height `6`.

All three bars could have height 1 or all three could have height 6. Those are valid results. The checker looks at many grids to distinguish occasional matches from a height accidentally reused every time.

In [ ]:
def random_height_bars():
    return None


In [ ]:
#@title Check Exercise 3

def check_exercise_3():
    return check_bar_generator(random_height_bars, 3)


make_check_button(check_exercise_3)


In [ ]:
show_samples(random_height_bars)


## Exercise 4: A random number of bars

Write `random_count_bars()`.

Keep the shared random color and the separate random heights from Exercise 3. Now also choose the number of bars from `1` through `5`. Keep the canvas height at `6`.

A useful order is:

1. Choose the shared color.
2. Choose `bar_count`.
3. Use `range(bar_count)` to make that many height choices.
4. Draw the result.

You do not need to choose a width separately: `draw_bars` calculates it from the number of heights.

**Predict:** What are the smallest and largest possible widths?

In [ ]:
def random_count_bars():
    return None


In [ ]:
#@title Check Exercise 4

def check_exercise_4():
    return check_bar_generator(random_count_bars, 4)


make_check_button(check_exercise_4)


In [ ]:
show_samples(random_count_bars)


## Exercise 5: Random canvas size

Write `random_size_bars()`.

Keep all the variation from Exercise 4, and choose the canvas height from `4` through `9`.

The canvas height and the number of bars are **separate random choices**. A short canvas can be wide, and a tall canvas can be narrow.

Every bar must fit on the chosen canvas. Choose the canvas height first, then use:

```python
random.randint(1, grid_height)
```

for each bar's height. The upper limit is now a variable, rather than a fixed number.

The shared color still varies from `1` through `9`, and the number of bars still varies from `1` through `5`.

In [ ]:
def random_size_bars():
    return None


In [ ]:
#@title Check Exercise 5

def check_exercise_5():
    return check_bar_generator(random_size_bars, 5)


make_check_button(check_exercise_5)


In [ ]:
show_samples(random_size_bars)


## What the checker can catch

If a check fails, read its first message and inspect a few samples.

- **Only one color appears:** Is the random call inside your function?
- **Every bar always matches its neighbor:** Is the height choice inside the bar loop?
- **The largest possible value never appears:** Did you remember that `randint` includes both endpoints?
- **Some bars do not fit:** Are you using the chosen canvas height as the upper limit?
- **The same grid appears every time:** Are you reusing an old result or restarting the random sequence inside the generator?

A repeated color or repeated grid by itself is normal. The checkers look at hundreds of results.

# Part 4: Repeat a random experiment

Python's random-number library is **pseudorandom**: it produces a sequence using an algorithm. A **seed** sets the starting point. Using the same seed and the same sequence of random calls lets you repeat an experiment, which is helpful when debugging.

Run the next cell. Both groups of die rolls will match because we restart at the same seed before each group.

The state-saving lines preserve your place in the random sequence before and after this demonstration. You do not need them inside your generator.

In [ ]:
# Reproducibility demonstration — provided.
saved_state = random.getstate()

random.seed(42)
first_rolls = []
for roll_number in range(6):
    first_rolls.append(random.randint(1, 6))

random.seed(42)
second_rolls = []
for roll_number in range(6):
    second_rolls.append(random.randint(1, 6))

random.setstate(saved_state)

print("First experiment: ", first_rolls)
print("Repeated experiment:", second_rolls)
print("Do they match?", first_rolls == second_rolls)


### A seed belongs before an experiment

If you set the same seed at the start of your generator on every call, you restart the sequence each time. That can make it return the same grid repeatedly.

For our exercises, leave seed-setting to the checker. When you later want to reproduce an experiment yourself, set a seed once **before** a series of generator calls.

### My observations

- **A property that can now vary:**
- **A property that every valid grid still shares:**
- **Why a repeated result can be normal:**
- **Why repeatedly setting the same seed is a problem:**

## Optional experiments

The five checked exercises are the core lesson. Choose an experiment if you have time:

1. **A smaller palette:** Use `random.choice([1, 2, 4])` to choose one item from a list of allowed colors. Make a new generator so the core exercises still use colors 1–9.
2. **Count colors:** Call `random_color()` 100 times and count how often a color of your choice appears. Repeat the experiment. Do the counts have to be identical?
3. **Make an ARC example:** Use `random_size_bars()` to create an input and a transformation from Week 5(b) to create its output. Keep the same transformation across several examples. Copy the needed transformation into your workspace if you want to try this.
4. **How many possibilities?** Exercise 2 has 9 possible grids. Exercise 3 has `9 * 6 * 6 * 6`. Explain why the three height choices are multiplied together.

## Wrap-up and reflection

Answer briefly:

- Which choice did you make once per grid? Which choice once per bar?
- What did each new exercise add to the variety of possible grids?
- How do the endpoints of `randint` differ from those of `range`?
- Why is testing one generated grid not enough?
- What can repeated sampling tell you, and what can it not prove?

### My reflection

- **Once per grid / once per bar:**
- **How the variety grew:**
- **Endpoints:**
- **Why many samples helped:**
- **One limit of the checker:**

## Share your completed notebook

When you finish:

1. Confirm that all five exercise checks pass.
2. Run the previews and inspect some of your generated grids.
3. Complete the reflection.
4. In Colab, click **Share**.
5. Add **michael.d.moffitt@gmail.com** as a Viewer or Commenter.
6. Send the share invite.

You have built a family of ARC-style inputs that can provide new examples for future lessons.